# Sesión 13 — Redes Neuronales Convolucionales
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo IV · Fundamentos de Deep Learning**

## Objetivos de aprendizaje

1. Comprender la operación de convolución y los sesgos inductivos que codifica (equivarianza traslacional, localidad).
2. Derivar la fórmula del tamaño de salida y comprender stride, padding y dilatación.
3. Construir una CNN 1-D desde cero en PyTorch para clasificación de arritmias en ECG.
4. Comprender las conexiones residuales (skip connections) de ResNet y por qué resuelven el problema de degradación.
5. Aplicar transfer learning desde modelos preentrenados en ImageNet a imágenes médicas.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | LeCun, Y. et al. (1998). Gradient-based learning applied to document recognition. *Proc. IEEE*, 86(11). — LeNet, el artículo fundacional. |
| ★★★ | He, K. et al. (2016). Deep residual learning for image recognition. *CVPR*. — ResNet. |
| ★★☆ | Kiranyaz, S. et al. (2016). 1-D convolutional neural networks and applications. *Mechanical Systems and Signal Processing*, 151. |
| ★★☆ | Hannun, A.Y. et al. (2019). Cardiologist-level arrhythmia detection and classification in ambulatory ECGs using a deep neural network. *Nature Medicine*, 25. |
| ★☆☆ | Guía CNN de CS231n: https://cs231n.github.io/convolutional-networks/ |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(7)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

## Parte 1 — La operación de convolución desde cero

Una convolución 1-D con filtro $\mathbf{f}$ de longitud $K$:
$$y[n] = (x * f)[n] = \sum_{k=0}^{K-1} x[n+k] \cdot f[k]$$

Propiedades clave: **compartición de pesos** (el mismo $\mathbf{f}$ se aplica en todas partes),
**campos receptivos locales**, **equivarianza traslacional**.

In [ ]:
# Simular un único latido de ECG
def latido_ecg_sintetico(length=250, fs=250, noise=0.05, rng_=None):
    rng_ = rng_ or np.random.default_rng()
    t = np.arange(length) / fs
    # Onda P, complejo QRS, onda T
    p     = 0.15 * np.exp(-((t - 0.12)**2) / (2*0.015**2))
    q     = -0.08 * np.exp(-((t - 0.20)**2) / (2*0.008**2))
    r     =  1.00 * np.exp(-((t - 0.22)**2) / (2*0.006**2))
    s     = -0.12 * np.exp(-((t - 0.24)**2) / (2*0.008**2))
    t_ond =  0.25 * np.exp(-((t - 0.38)**2) / (2*0.025**2))
    latido = p + q + r + s + t_ond + rng_.normal(0, noise, length)
    return latido

latido = latido_ecg_sintetico(rng_=rng)
t_latido = np.linspace(0, 1, 250)

# Ejemplos de filtros
filtros = {
    'Detector de bordes\n(derivada)':    np.array([-1., 0., 1.]),
    'Suavizado\n(Gaussiano)':            np.exp(-np.linspace(-2, 2, 11)**2),
    'Detector de QRS\n(filtro acoplado)': latido_ecg_sintetico(length=40, noise=0)[::-1],
}

fig, axes = plt.subplots(len(filtros)+1, 1, figsize=(13, 9), sharex=False)

axes[0].plot(t_latido, latido, 'steelblue', lw=1.5)
axes[0].set(ylabel='mV', title='Latido de ECG sintético (señal de entrada)')

for ax, (nombre, filt) in zip(axes[1:], filtros.items()):
    filt_norm = filt / (np.sum(np.abs(filt)) + 1e-8)
    respuesta = np.convolve(latido, filt_norm, mode='same')
    ax.plot(t_latido, latido,    'steelblue', lw=0.8, alpha=0.4, label='Entrada')
    ax.plot(t_latido, respuesta, 'tomato',    lw=2,   label='Respuesta del filtro')
    ax.set(ylabel='Respuesta', title=f'Filtro: {nombre}')
    ax.legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Tiempo (s)')
plt.suptitle('Convolución en una señal de ECG 1-D — tres ejemplos de filtro', y=1.01)
plt.tight_layout()
plt.show()

# Fórmula del tamaño de salida
print('\nFórmula del tamaño de salida:')
print('  L_out = piso((L_in + 2·P - D·(K-1) - 1) / S + 1)')
print('  L_in=250, K=11, P=0, D=1, S=1 →', (250 + 0 - 1*(11-1) - 1)//1 + 1)
print('  L_in=250, K=11, P=5, D=1, S=1 →', (250 + 10 - 1*(11-1) - 1)//1 + 1, '(padding "same")')
print('  L_in=250, K=11, P=0, D=1, S=2 →', (250 + 0 - 1*(11-1) - 1)//2 + 1, '(stride 2)')

## Parte 2 — CNN 1-D para clasificación de arritmias en ECG (estilo PhysioNet MIT-BIH)

Construimos una red inspirada en Hannun et al. (2019) — convoluciones 1-D operando
directamente sobre muestras crudas de ECG, sin necesidad de características diseñadas a mano.

In [ ]:
# ── Simular segmentos de latidos estilo MIT-BIH (sustituir por wfdb para datos reales) ──
# 5 clases AAMI: N, S, V, F, Q
# Cada latido: 250 muestras a 360 Hz (~0.7s, centrado en el pico R)
n_por_clase = {'N': 1000, 'S': 150, 'V': 200, 'F': 50, 'Q': 40}
seg_len = 250
clases  = list(n_por_clase.keys())

def generar_clase_latido(tipo_latido, n, rng_):
    latidos = []
    for _ in range(n):
        b = latido_ecg_sintetico(length=seg_len, noise=0.04, rng_=rng_)
        if tipo_latido == 'V':   # QRS ancho, sin onda P
            t = np.arange(seg_len) / 250
            b = 0.9*np.exp(-((t-0.22)**2)/(2*0.018**2)) + \
                -0.5*np.exp(-((t-0.28)**2)/(2*0.015**2)) + \
                0.2*np.exp(-((t-0.40)**2)/(2*0.030**2)) + \
                rng_.normal(0, 0.05, seg_len)
        elif tipo_latido == 'S': # Estrecho, prematuro
            b = 0.8 * b
        elif tipo_latido == 'F': # Fusión
            b = 0.6*b + 0.4*generar_clase_latido('V', 1, rng_)[0]
        elif tipo_latido == 'Q': # Desconocido / artefacto
            b = rng_.normal(0, 0.4, seg_len)
        latidos.append(b)
    return np.array(latidos)

X_segs_list, y_segs_list = [], []
for k, (cls, n) in enumerate(n_por_clase.items()):
    X_segs_list.append(generar_clase_latido(cls, n, rng))
    y_segs_list.append(np.full(n, k))

X_segs = np.vstack(X_segs_list).astype(np.float32)  # (N, 250)
y_segs = np.hstack(y_segs_list).astype(np.int64)

# Normalizar cada latido independientemente (z-score)
X_segs = (X_segs - X_segs.mean(axis=1, keepdims=True)) / \
          (X_segs.std(axis=1, keepdims=True) + 1e-8)

# Añadir dimensión de canal: (N, 1, 250)
X_segs = X_segs[:, np.newaxis, :]

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_segs, y_segs, test_size=0.25, stratify=y_segs, random_state=0)

print(f'Dataset: {X_segs.shape[0]} latidos, forma {X_segs.shape}')
print(f'Distribución de clases: {dict(zip(clases, np.bincount(y_segs)))}')

In [ ]:
class ECG_CNN(nn.Module):
    """
    CNN 1-D para clasificación de latidos ECG en 5 clases.
    Arquitectura: bloques Conv (Conv→BN→ReLU→Pool) + cabeza FC.
    """
    def __init__(self, n_classes=5, seg_len=250):
        super().__init__()
        self.features = nn.Sequential(
            # Bloque 1
            nn.Conv1d(1,  32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),             # 250 → 125
            # Bloque 2
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),             # 125 → 62
            # Bloque 3
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.MaxPool1d(2),             # 62 → 31
            # Bloque 4
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(8),     # → 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*8, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

    def feature_maps(self, x):
        """Retorna los mapas de características intermedios para visualización."""
        maps = []
        for layer in self.features:
            x = layer(x)
            if isinstance(layer, nn.ReLU):
                maps.append(x.detach())
        return maps

# Resumen del modelo
model_cnn = ECG_CNN(n_classes=5).to(device)
total_params = sum(p.numel() for p in model_cnn.parameters())
entrenables  = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f'ECG-CNN: {total_params:,} parámetros totales, {entrenables:,} entrenables')

# Verificación rápida del forward pass
dummy = torch.randn(4, 1, 250).to(device)
out   = model_cnn(dummy)
print(f'Forward pass: entrada {tuple(dummy.shape)} → salida {tuple(out.shape)}')

In [ ]:
# ── Bucle de entrenamiento ─────────────────────────────────────────────────────
Xtr_t = torch.tensor(X_tr_s, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_s, dtype=torch.long)
Xte_t = torch.tensor(X_te_s, dtype=torch.float32)
yte_t = torch.tensor(y_te_s, dtype=torch.long)

# Pesos de clase para el desbalance
counts     = np.bincount(y_tr_s)
class_wts  = torch.tensor(1.0 / counts, dtype=torch.float32).to(device)
class_wts /= class_wts.sum()

train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t),
                           batch_size=64, shuffle=True)

model_cnn = ECG_CNN(n_classes=5).to(device)
optimizer  = optim.Adam(model_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
criterion  = nn.CrossEntropyLoss(weight=class_wts)

train_losses, val_accs = [], []
n_epochs = 30

for epoch in range(n_epochs):
    model_cnn.train()
    epoch_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model_cnn(Xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_cnn.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    model_cnn.eval()
    with torch.no_grad():
        logits_val = model_cnn(Xte_t.to(device))
        preds_val  = logits_val.argmax(1).cpu().numpy()
        acc_val    = (preds_val == y_te_s).mean()
    train_losses.append(epoch_loss / len(train_loader))
    val_accs.append(acc_val)
    if (epoch+1) % 5 == 0:
        print(f'Época {epoch+1:3d}/{n_epochs}  pérdida={train_losses[-1]:.4f}  acc_val={acc_val:.4f}')

# Evaluación final
model_cnn.eval()
with torch.no_grad():
    logits_te = model_cnn(Xte_t.to(device)).cpu().numpy()
    proba_te  = F.softmax(torch.tensor(logits_te), dim=1).numpy()
    preds_te  = logits_te.argmax(1)

print('\n' + classification_report(y_te_s, preds_te, target_names=clases))

## Parte 3 — Visualización de mapas de características

In [ ]:
# Visualizar lo que aprendió la primera capa convolucional
model_cnn.eval()

# Graficar filtros aprendidos de Conv1 (32 filtros de longitud 11)
w1 = model_cnn.features[0].weight.data.cpu().numpy()  # (32, 1, 11)

fig, axes = plt.subplots(4, 8, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    if i < 32:
        ax.plot(w1[i, 0], color='steelblue', lw=1.5)
        ax.axhline(0, color='gray', lw=0.5, ls='--')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_facecolor('#f8f8f8')
    else:
        ax.set_visible(False)

plt.suptitle('Filtros Conv1 aprendidos (32 × longitud-11)\n'
              'Cada filtro detecta un micropatrón distinto del ECG', y=1.01)
plt.tight_layout()
plt.show()

# Visualizar mapas de características para un latido Normal y un PVC
fig, axes = plt.subplots(2, 4, figsize=(15, 5))
tipos_latido_mostrar = [0, 2]   # Normal (0) y PVC/V (2)

for row, cls_idx in enumerate(tipos_latido_mostrar):
    muestra_latido = torch.tensor(
        X_te_s[y_te_s == cls_idx][0:1], dtype=torch.float32).to(device)
    with torch.no_grad():
        fmaps = model_cnn.feature_maps(muestra_latido)

    axes[row, 0].plot(muestra_latido.cpu().squeeze().numpy(), lw=1.5, color='navy')
    axes[row, 0].set(title=f'Entrada: {clases[cls_idx]}', ylabel='Amplitud (norm.)')

    for col, fmap in enumerate(fmaps[:3], start=1):
        fm = fmap.cpu().squeeze().numpy()   # (C, L)
        axes[row, col].imshow(fm[:16], aspect='auto', cmap='RdBu_r')
        axes[row, col].set(title=f'Mapa de características bloque {col} (16/C mostrados)',
                            xlabel='Tiempo →', ylabel='Índice de filtro')

plt.suptitle('Evolución de mapas de características — Normal vs PVC\n'
              'La CNN aprende a detectar diferencias morfológicas automáticamente', y=1.01)
plt.tight_layout()
plt.show()

## Parte 4 — Conexiones residuales de ResNet

Un **bloque residual** aprende $\mathcal{F}(\mathbf{x}) = H(\mathbf{x}) - \mathbf{x}$ (el residuo),
por lo que el mapeo completo es $H(\mathbf{x}) = \mathcal{F}(\mathbf{x}) + \mathbf{x}$.

Esto resuelve el **problema de degradación**: añadir más capas a una red simple puede
*aumentar* el error de entrenamiento. Con conexiones residuales, las capas extra siempre
pueden aprender la identidad — en el peor caso, $\mathcal{F}(\mathbf{x}) \to 0$.

In [ ]:
class ResBlock1D(nn.Module):
    """Bloque residual 1-D con submuestreo opcional."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 5, stride=stride, padding=2, bias=False),
            nn.BatchNorm1d(out_channels), nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(out_channels, out_channels, 5, padding=2, bias=False),
            nn.BatchNorm1d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        self.relu_out = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = out + self.shortcut(x)   # ← conexión residual
        return self.relu_out(out)


class ResNet_ECG(nn.Module):
    """ResNet 1-D compacta para clasificación de ECG."""
    def __init__(self, n_classes=5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, 15, padding=7, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.layer1 = ResBlock1D(32,  64,  stride=2)
        self.layer2 = ResBlock1D(64,  128, stride=2)
        self.layer3 = ResBlock1D(128, 128, stride=2)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.head(x)


model_res = ResNet_ECG().to(device)
res_params = sum(p.numel() for p in model_res.parameters())
print(f'ResNet-ECG: {res_params:,} parámetros')

# Entrenar ResNet
optimizer_r = optim.Adam(model_res.parameters(), lr=1e-3, weight_decay=1e-4)
sched_r     = optim.lr_scheduler.CosineAnnealingLR(optimizer_r, T_max=30)
hist_res    = []

for epoch in range(30):
    model_res.train()
    ep_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer_r.zero_grad()
        loss = criterion(model_res(Xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_res.parameters(), 1.0)
        optimizer_r.step()
        ep_loss += loss.item()
    sched_r.step()
    hist_res.append(ep_loss / len(train_loader))

model_res.eval()
with torch.no_grad():
    preds_res = model_res(Xte_t.to(device)).argmax(1).cpu().numpy()
acc_res = (preds_res == y_te_s).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, lw=2, label=f'CNN simple  (acc val={val_accs[-1]:.3f})')
ax.plot(hist_res,     lw=2, label=f'ResNet      (acc val={acc_res:.3f})')
ax.set(xlabel='Época', ylabel='Pérdida de entrenamiento',
       title='CNN simple vs ResNet — curvas de entrenamiento')
ax.legend()
plt.tight_layout()
plt.show()

## Parte 5 — Transfer learning para imágenes médicas

Los modelos preentrenados en ImageNet proporcionan características de bajo nivel
potentes (bordes, texturas, gradientes) que transfieren bien a imágenes médicas —
incluso cuando el dominio fuente es muy distinto.

In [ ]:
import torchvision.models as tv_models

# Simular una tarea de clasificación binaria de imágenes médicas
# (ej.: dermatoscopia: melanoma vs. nevus benigno)
n_imagenes = 400
img_size   = 128

# Imágenes sintéticas de 3 canales
X_img = rng.random((n_imagenes, 3, img_size, img_size)).astype(np.float32)
# Clase positiva: intensidad ligeramente mayor en el centro (simulando lesión)
y_img = rng.integers(0, 2, n_imagenes).astype(np.int64)
for i in np.where(y_img == 1)[0]:
    cx, cy = img_size//2, img_size//2
    Y, X = np.ogrid[:img_size, :img_size]
    mask = (X-cx)**2 + (Y-cy)**2 < (img_size//4)**2
    X_img[i, :, mask] += rng.normal(0.3, 0.1)
X_img = np.clip(X_img, 0, 1)

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_img, y_img, test_size=0.25, stratify=y_img, random_state=0)

Xi_tr_t = torch.tensor(Xi_tr)
yi_tr_t = torch.tensor(yi_tr)
Xi_te_t = torch.tensor(Xi_te)

img_loader = DataLoader(TensorDataset(Xi_tr_t, yi_tr_t), batch_size=32, shuffle=True)

def crear_modelo_transfer(congelar_backbone=True):
    backbone = tv_models.resnet18(weights='IMAGENET1K_V1')
    if congelar_backbone:
        for param in backbone.parameters():
            param.requires_grad = False
    # Reemplazar la capa final
    in_feat = backbone.fc.in_features
    backbone.fc = nn.Linear(in_feat, 2)
    return backbone

resultados_tl = {}
for modo, congelar in [('Backbone congelado', True), ('Ajuste fino completo', False)]:
    model_tl = crear_modelo_transfer(congelar).to(device)
    entrenables_tl = sum(p.numel() for p in model_tl.parameters() if p.requires_grad)
    lr_tl = 1e-3 if congelar else 1e-4
    opt_tl = optim.Adam(filter(lambda p: p.requires_grad, model_tl.parameters()), lr=lr_tl)
    crit_tl = nn.CrossEntropyLoss()

    for ep in range(10):
        model_tl.train()
        for Xb, yb in img_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt_tl.zero_grad()
            crit_tl(model_tl(Xb), yb).backward()
            opt_tl.step()

    model_tl.eval()
    with torch.no_grad():
        p_tl = F.softmax(model_tl(Xi_te_t.to(device)), dim=1)[:,1].cpu().numpy()
    auroc_tl = roc_auc_score(yi_te, p_tl)
    resultados_tl[modo] = {'AUROC': auroc_tl, 'entrenables': entrenables_tl}
    print(f'{modo:25s}  entrenables={entrenables_tl:,}  AUROC={auroc_tl:.3f}')

print('\nIdea clave: el backbone congelado (extracción de características) es más rápido')
print('y a menudo suficiente cuando los datos médicos etiquetados son escasos.')
print('El ajuste fino completo ayuda cuando se dispone de más datos.')

## ✏️ Ejercicios

1. **Cálculo del campo receptivo.** Para la CNN de 4 bloques de la Parte 2, calcula el
   tamaño del campo receptivo a la salida de cada bloque. Demuestra que después de 4
   bloques de max-pooling con stride 2 y convoluciones de tamaño de kernel 7, una sola
   neurona de salida "ve" la mayor parte de la entrada de 250 muestras. ¿Por qué importa
   esto para la detección de arritmias?

2. **Aumentación de datos para ECG.** Implementa tres estrategias de aumentación:
   (a) escalado aleatorio de amplitud ×[0.8, 1.2], (b) desplazamiento temporal aleatorio
   ±10 muestras, (c) adición de ruido gaussiano. Aplícalas en el loader de entrenamiento
   y mide el efecto en el AUROC cuando los datos de entrenamiento se reducen al 20% del original.

3. **Datos reales de MIT-BIH.** Instala `wfdb` (`pip install wfdb`) y descarga la base
   de datos MIT-BIH. Extrae segmentos de 250 muestras centrados en picos R anotados
   (5 clases AAMI). Vuelve a ejecutar el pipeline de entrenamiento y compara el
   rendimiento con el dataset simulado.

4. **CNN multi-derivación.** Extiende la ECG-CNN para aceptar entrada de 2 derivaciones
   (modifica `in_channels=2`). Simula un dataset de 2 derivaciones y compara el AUROC
   de derivación única vs 2 derivaciones.

5. *(Desafío)* **Convoluciones causales dilatadas.** Las convoluciones dilatadas estilo
   WaveNet permiten campos receptivos exponencialmente grandes sin perder resolución.
   Implementa una CNN 1-D dilatada con tasas de dilatación [1, 2, 4, 8, 16, 32] y
   compárala con la CNN estándar en clasificación de segmentos largos de ECG
   (ventanas de 5 segundos en lugar de segmentos de latido individual).

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| MIT-BIH Arrhythmia | https://physionet.org/content/mitdb/ | `pip install wfdb` |
| PhysioNet CinC 2017 | https://physionet.org/content/challenge-2017/ | Detección de FA en ECG de una derivación |
| ISIC Dermoscopy | https://www.isic-archive.com | Clasificación de lesiones cutáneas, 25k imágenes |
| ChestX-ray14 | https://nihcc.app.box.com/v/ChestXray-NIHCC | 112k radiografías de tórax, 14 patologías |